In [1]:
import torch
from dinosaw.utils import do_2D_pca
from dinosaw.helpers import ModelTypes, get_models, get_features, model_names

import numpy as np
from PIL import Image

import matplotlib.pyplot as plt

SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)

half = False
torch.cuda.empty_cache()
DEVICE = 'cuda:1'

flash attention installed


In [2]:
enabled_models: tuple[ModelTypes, ...] = ('dv2', 'dvt', 'alibi_dv2', 'alibi_dv2_cb', 'alibi_dv2_h', 'nope')

In [3]:
names: dict[str, str] = {'dv2': 'DINOv2', 'dvt': "DVT", "alibi": "ALiBi-Dv2", "alibi_h": "ALiBi(H)-Dv2", 'nope': 'NoPE'}
models = get_models(enabled_models, '../../trained_models', DEVICE, False)



features = {key: [] for key in enabled_models}
features_reduced: dict[ModelTypes, list[np.ndarray]] = {key:[] for key in enabled_models}

image_lengths = [256, 512, 768, 1024, 1536]
# image_names = ('diff_shapes_518.png', 'plant.png', 'bulldog_518.png', 'ni_superalloy.png', 'needle_block_.jpg', 'cell.jpg', 'EBC.png', 'LFP.jpg')
image_names = ('diff_shapes_518.png', 'plant.png', 'bulldog_518.png')
img_results = {name: [{key: [] for key in enabled_models} for _ in image_lengths] for name in image_names}


In [4]:
for image_name in image_names:
    pil_img = Image.open(f'data/length_generalization/{image_name}').convert('RGB')
    for i, l in enumerate(image_lengths):
        pil_img = pil_img.resize((l, l))
        for model_key in enabled_models:
            feats = get_features(models[model_key], pil_img, channel_last=False, channel_blank=False, device=DEVICE)
            features_reduced = do_2D_pca(feats, n_components=3, post_norm='minmax')
            img_results[image_name][i][model_key] = features_reduced
        

In [6]:
%%capture
fig, axs = plt.subplots(nrows=len(image_names) * len(enabled_models), ncols=len(image_lengths), figsize=(15, 3 * len(image_names) * len(enabled_models)))

for k, image_name in enumerate(image_names):
    for j, model_key in enumerate(enabled_models):
        for i, l in enumerate(image_lengths):
            ax = axs[k * len(enabled_models) + j, i]
            ax.imshow(img_results[image_name][i][model_key])
            ax.set_xticks([])
            ax.set_yticks([])
            if j == 0:
                ax.set_title(f'Length: {l}px', fontsize=10)
            if i == 0:
                ax.set_ylabel(f'{model_names[model_key]}', fontsize=10)
